In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel, pipeline
import torch
from sageattention import sageattn
import re
import gc
import json
import accelerate
from datasets import load_from_disk
import time
import os
from torch import Tensor
import numpy as np
import random

import torch.nn.functional as F
F.scaled_dot_product_attention = sageattn
model_name = "Meta-Llama-3.1-8B-Instruct"
model_path = "/ssd/data/data/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/5206a32e0bd3067aef1ce90f5528ade7d866253f"
tokenizer = AutoTokenizer.from_pretrained(model_path , trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto", 
    trust_remote_code=True,
    device_map="auto",
    attn_implementation="flash_attention_2"
)
model.eval()
torch.manual_seed(42)

/ssd/data/data/dyf/env/miniconda3/envs/sage/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.32it/s]


In [2]:
from tests.modify_flash_attn_triton import set_flash_attn_triton_llama
set_flash_attn_triton_llama(model, smooth_k=False)

In [7]:
prompt = "Please explain the principle of gravity to me"
# prompt = "The sky is blue and the grass is" 
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs.input_ids.to(model.device)
attention_mask = inputs["attention_mask"].to(model.device)
print(f"input_ids.shape: {input_ids.shape}")
output = model.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer.eos_token_id, max_new_tokens = 100, use_cache=True, do_sample=False)
result = tokenizer.decode(output[0].tolist()[input_ids.shape[1]:], skip_special_tokens=True)
print(result)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


input_ids.shape: torch.Size([1, 9])
.
Gravity is a fundamental force of nature that causes objects with mass to attract each other. The principle of gravity is based on the idea that every object in the universe has mass, and that mass is a measure of the amount of matter in an object. The more massive an object is, the stronger its gravitational pull.
The principle of gravity was first described by Sir Isaac Newton in his groundbreaking work "Philosophiæ Naturalis Principia Mathematica" in 1687. Newton's law of
